In [1]:
# Core libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Statistical analysis
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.graphics.factorplots import (
    interaction_plot,
)

In [2]:
# Assemble the trimmed two-AA mixture dataset.

# DATA_DIR = Path(".")
DATA_DIR = Path(
    "/Volumes/JAEONCHO/ADIC-AA/g_CD data"
)

TRIM_EVENT_DIR = (
    DATA_DIR / "trim_events_g_CD_mix"
)

TRIM_SUB_EVENT_DIR = (
    DATA_DIR / "trim_sub_events_g_CD_mix"
)

BASELINE_DIR = (
    DATA_DIR / "final_baseline_info_g_CD_mix"
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assembled_datasets = []

for sub_event_file in sorted(
    TRIM_SUB_EVENT_DIR.glob("*.csv")
):
    recording_id = (
        sub_event_file.stem.removesuffix("_sub")
    )

    mixture_label = (
        recording_id.split("_")[0]
    )

    event_file = (
        TRIM_EVENT_DIR
        / f"{recording_id}.csv"
    )

    baseline_file = (
        BASELINE_DIR
        / f"{recording_id}_final.csv"
    )

    if not event_file.exists():
        raise FileNotFoundError(
            f"Missing trimmed event file: "
            f"{event_file.name}"
        )

    if not baseline_file.exists():
        raise FileNotFoundError(
            f"Missing baseline file: "
            f"{baseline_file.name}"
        )

    sub_events = pd.read_csv(
        sub_event_file
    )

    first_level_events = pd.read_csv(
        event_file
    )

    baseline_data = pd.read_csv(
        baseline_file
    )

    if baseline_data.empty:
        raise ValueError(
            f"Baseline file is empty: "
            f"{baseline_file.name}"
        )

    open_pore_rms = (
        baseline_data["rms"].iloc[0]
    )

    matched_rows = []

    for _, sub_event in sub_events.iterrows():
        parent_event = first_level_events.loc[
            (
                first_level_events["start_time_s"]
                <= sub_event["sub_start_time_s"]
            )
            & (
                first_level_events["end_time_s"]
                >= sub_event["sub_end_time_s"]
            )
        ]

        if parent_event.empty:
            continue

        parent_event = parent_event.iloc[0]

        event_id = (
            f"{recording_id}_"
            f"{int(parent_event['event_index'])}"
        )

        matched_rows.append({
            "sub_post_interaction_current":
                sub_event[
                    "sub_post_interaction_current"
                ],

            # I1
            "first_level_current":
                sub_event["sub_mode_baseline"],

            # σ1
            "first_level_stdev":
                parent_event["blockage_stdev"],

            # σ0
            "open_pore_rms":
                open_pore_rms,

            "dwell_time_us":
                parent_event["dwell_time_us"],

            # Parent first-level blockage
            "event_id":
                event_id,

            # Single-pore recording
            "dataset_id":
                recording_id,

            # Two-AA mixture identity
            "mixture_label":
                mixture_label,
        })

    if matched_rows:
        assembled_datasets.append(
            pd.DataFrame(matched_rows)
        )

if not assembled_datasets:
    raise ValueError(
        "No matched mixture observations were found."
    )

df = pd.concat(
    assembled_datasets,
    ignore_index=True,
)

required_columns = [
    "sub_post_interaction_current",
    "first_level_current",
    "first_level_stdev",
    "open_pore_rms",
    "dwell_time_us",
    "event_id",
    "dataset_id",
    "mixture_label",
]

if df[required_columns].isna().any().any():
    missing_columns = (
        df[required_columns]
        .columns[
            df[required_columns]
            .isna()
            .any()
        ]
        .tolist()
    )

    raise ValueError(
        f"Missing values detected in: "
        f"{missing_columns}"
    )

if (df["open_pore_rms"] <= 0).any():
    raise ValueError(
        "All open-pore RMS values must be positive."
    )

print(
    f"Mixture identities: "
    f"{df['mixture_label'].nunique()}"
)

print(
    f"Secondary events: "
    f"{len(df):,}"
)

print(
    f"First-level events: "
    f"{df['event_id'].nunique():,}"
)

print(
    f"Single-pore recordings: "
    f"{df['dataset_id'].nunique():,}"
)

display(df.head())

Mixture identities: 20
Secondary events: 94,867
First-level events: 7,052
Single-pore recordings: 106


,sub_post_interaction_current,first_level_current,first_level_stdev,open_pore_rms,dwell_time_us,event_id,dataset_id,mixture_label
0,120.849610,128.78418,12.573928,4.952532,12050.0,DP_g_CD_250829__0001_0,DP_g_CD_250829__0001,DP
1,117.255320,128.78418,12.573928,4.952532,12050.0,DP_g_CD_250829__0001_0,DP_g_CD_250829__0001,DP
2,115.173340,128.78418,12.573928,4.952532,12050.0,DP_g_CD_250829__0001_0,DP_g_CD_250829__0001,DP
3,115.478516,128.78418,12.573928,4.952532,12050.0,DP_g_CD_250829__0001_0,DP_g_CD_250829__0001,DP
4,117.340090,128.78418,12.573928,4.952532,12050.0,DP_g_CD_250829__0001_0,DP_g_CD_250829__0001,DP


In [3]:
# Construct the four features used in the
# two-way ANOVA analysis.

FEATURE_NAMES = [
    "Baseline Ratio",
    "Z Score",
    "Per-event Impact",
    "Per-time Impact",
]


# Calculate recording-level aggregate values.
secondary_event_count = (
    df.groupby("dataset_id")
    .size()
    .rename(
        "secondary_event_count"
    )
)

first_level_dwell_time = (
    df[
        [
            "dataset_id",
            "event_id",
            "dwell_time_us",
        ]
    ]
    .drop_duplicates(subset="event_id")
    .groupby("dataset_id")[
        "dwell_time_us"
    ]
    .sum()
    .rename(
        "total_first_level_dwell_time"
    )
)

recording_aggregates = pd.concat(
    [
        secondary_event_count,
        first_level_dwell_time,
    ],
    axis=1,
)

recording_aggregates[
    "secondary_event_frequency"
] = (
    recording_aggregates[
        "secondary_event_count"
    ]
    / recording_aggregates[
        "total_first_level_dwell_time"
    ]
)


if (
    recording_aggregates.isna().any().any()
    or (
        recording_aggregates[
            "secondary_event_count"
        ]
        <= 0
    ).any()
    or (
        recording_aggregates[
            "total_first_level_dwell_time"
        ]
        <= 0
    ).any()
    or (
        recording_aggregates[
            "secondary_event_frequency"
        ]
        <= 0
    ).any()
):
    raise ValueError(
        "Invalid recording-level aggregate values."
    )


feature_data = df.copy()

feature_data[
    "secondary_event_count"
] = feature_data["dataset_id"].map(
    recording_aggregates[
        "secondary_event_count"
    ]
)

feature_data[
    "total_first_level_dwell_time"
] = feature_data["dataset_id"].map(
    recording_aggregates[
        "total_first_level_dwell_time"
    ]
)

feature_data[
    "secondary_event_frequency"
] = feature_data["dataset_id"].map(
    recording_aggregates[
        "secondary_event_frequency"
    ]
)


# Local features
feature_data["Baseline Ratio"] = (
    feature_data[
        "sub_post_interaction_current"
    ]
    / feature_data[
        "first_level_current"
    ]
)

feature_data["Z Score"] = (
    (
        feature_data[
            "sub_post_interaction_current"
        ]
        - feature_data[
            "first_level_current"
        ]
    )
    / feature_data[
        "open_pore_rms"
    ]
)


# Global features
feature_data["Per-event Impact"] = (
    feature_data["Z Score"] ** 2
    / feature_data[
        "secondary_event_count"
    ]
)

feature_data["Per-time Impact"] = (
    feature_data["Z Score"] ** 2
    / feature_data[
        "secondary_event_frequency"
    ]
)


feature_values = (
    feature_data[
        FEATURE_NAMES
    ]
    .to_numpy()
)

if not np.isfinite(feature_values).all():
    raise ValueError(
        "Nonfinite values detected in the "
        "ANOVA features."
    )


# Retain the columns needed for the analysis
# and for tracing each observation to its source.
anova_data = feature_data[
    [
        "dataset_id",
        "event_id",
        "mixture_label",
        *FEATURE_NAMES,
    ]
].copy()


anova_data.to_csv(
    OUTPUT_DIR
    / "two_way_ANOVA_input.csv",
    index=False,
)


print(
    f"ANOVA observations: "
    f"{len(anova_data):,}"
)

print(
    f"Mixture identities: "
    f"{anova_data['mixture_label'].nunique()}"
)

display(anova_data.head())

ANOVA observations: 94,867
Mixture identities: 20


,dataset_id,event_id,mixture_label,Baseline Ratio,Z Score,Per-event Impact,Per-time Impact
0,DP_g_CD_250829__0001,DP_g_CD_250829__0001_0,DP,0.938389,-1.602124,0.012165,7784.827842
1,DP_g_CD_250829__0001,DP_g_CD_250829__0001_0,DP,0.910479,-2.327872,0.025682,16435.198425
2,DP_g_CD_250829__0001,DP_g_CD_250829__0001_0,DP,0.894313,-2.748259,0.035796,22907.203737
3,DP_g_CD_250829__0001,DP_g_CD_250829__0001_0,DP,0.896682,-2.686639,0.034209,21891.490180
4,DP_g_CD_250829__0001,DP_g_CD_250829__0001_0,DP,0.911137,-2.310755,0.025306,16194.395825


In [4]:
# Define all AA pairs included in the
# two-way ANOVA analysis.

ANOVA_PAIRS = [
    {
        "pair": "SL",
        "factor_1": "S",
        "factor_2": "L",
        "figure_group": "main",
    },
    {
        "pair": "DP",
        "factor_1": "D",
        "factor_2": "P",
        "figure_group": "supplementary",
    },
    {
        "pair": "DT",
        "factor_1": "D",
        "factor_2": "T",
        "figure_group": "supplementary",
    },
    {
        "pair": "GH",
        "factor_1": "G",
        "factor_2": "H",
        "figure_group": "supplementary",
    },
    {
        "pair": "HM",
        "factor_1": "H",
        "factor_2": "M",
        "figure_group": "supplementary",
    },
    {
        "pair": "LD",
        "factor_1": "L",
        "factor_2": "D",
        "figure_group": "supplementary",
    },
    {
        "pair": "MD",
        "factor_1": "M",
        "factor_2": "D",
        "figure_group": "supplementary",
    },
    {
        "pair": "MV",
        "factor_1": "M",
        "factor_2": "V",
        "figure_group": "supplementary",
    },
    {
        "pair": "SD",
        "factor_1": "S",
        "factor_2": "D",
        "figure_group": "supplementary",
    },
    {
        "pair": "TV",
        "factor_1": "T",
        "factor_2": "V",
        "figure_group": "supplementary",
    },
    {
        "pair": "VD",
        "factor_1": "V",
        "factor_2": "D",
        "figure_group": "supplementary",
    },
]


# Validate the pair definitions.
defined_pair_names = [
    pair_definition["pair"]
    for pair_definition in ANOVA_PAIRS
]

if len(defined_pair_names) != len(
    set(defined_pair_names)
):
    raise ValueError(
        "Duplicate ANOVA pair names were detected."
    )

for pair_definition in ANOVA_PAIRS:
    expected_pair_name = (
        pair_definition["factor_1"]
        + pair_definition["factor_2"]
    )

    if (
        pair_definition["pair"]
        != expected_pair_name
    ):
        raise ValueError(
            f"Pair-name mismatch for "
            f"{pair_definition}."
        )


print(
    f"Number of ANOVA pairs: "
    f"{len(ANOVA_PAIRS)}"
)

print(
    "Main-figure pair:",
    [
        pair["pair"]
        for pair in ANOVA_PAIRS
        if pair["figure_group"] == "main"
    ],
)

print(
    "Supplementary pairs:",
    [
        pair["pair"]
        for pair in ANOVA_PAIRS
        if pair["figure_group"]
        == "supplementary"
    ],
)

Number of ANOVA pairs: 11
Main-figure pair: ['SL']
Supplementary pairs: ['DP', 'DT', 'GH', 'HM', 'LD', 'MD', 'MV', 'SD', 'TV', 'VD']


In [5]:
# Perform two-way ANOVA and generate interaction
# plots for all defined AA pairs and features.

ANOVA_OUTPUT_DIR = (
    OUTPUT_DIR / "two_way_ANOVA"
)

ANOVA_TABLE_DIR = (
    ANOVA_OUTPUT_DIR / "tables"
)

MAIN_FIGURE_DIR = (
    ANOVA_OUTPUT_DIR / "main_figures"
)

SUPPLEMENTARY_FIGURE_DIR = (
    ANOVA_OUTPUT_DIR
    / "supplementary_figures"
)

for output_directory in [
    ANOVA_TABLE_DIR,
    MAIN_FIGURE_DIR,
    SUPPLEMENTARY_FIGURE_DIR,
]:
    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


plt.rcParams["font.family"] = "Arial"

all_anova_results = []
all_factor_counts = []


for pair_definition in ANOVA_PAIRS:
    pair_name = (
        pair_definition["pair"]
    )

    factor_1_aa = (
        pair_definition["factor_1"]
    )

    factor_2_aa = (
        pair_definition["factor_2"]
    )

    figure_group = (
        pair_definition["figure_group"]
    )

    pair_data = anova_data.copy()

    pair_data["factor_1_present"] = (
        pair_data[
            "mixture_label"
        ]
        .str.contains(
            factor_1_aa,
            regex=False,
        )
        .astype(int)
    )

    pair_data["factor_2_present"] = (
        pair_data[
            "mixture_label"
        ]
        .str.contains(
            factor_2_aa,
            regex=False,
        )
        .astype(int)
    )


    # Confirm that all four presence/absence
    # combinations are represented.
    factor_counts = (
        pair_data.groupby(
            [
                "factor_1_present",
                "factor_2_present",
            ]
        )
        .size()
        .rename("observation_count")
        .reset_index()
        .sort_values(
            [
                "factor_1_present",
                "factor_2_present",
            ]
        )
        .reset_index(drop=True)
    )

    if len(factor_counts) != 4:
        raise ValueError(
            f"All four presence/absence "
            f"combinations are required for "
            f"pair {pair_name}."
        )

    factor_counts.insert(
        0,
        "pair",
        pair_name,
    )

    factor_counts.insert(
        1,
        "factor_1_AA",
        factor_1_aa,
    )

    factor_counts.insert(
        2,
        "factor_2_AA",
        factor_2_aa,
    )

    factor_counts.insert(
        3,
        "figure_group",
        figure_group,
    )

    all_factor_counts.append(
        factor_counts
    )


    for feature_name in FEATURE_NAMES:
        print(
            f"Running {pair_name}: "
            f"{feature_name}",
            flush=True,
        )

        formula = (
            f'Q("{feature_name}") ~ '
            f'C(factor_1_present) '
            f'+ C(factor_2_present) '
            f'+ C(factor_1_present):'
            f'C(factor_2_present)'
        )

        model = ols(
            formula,
            data=pair_data,
        ).fit()

        # Type II two-way ANOVA
        anova_table = sm.stats.anova_lm(
            model,
            typ=2,
        )

        anova_table = (
            anova_table
            .reset_index()
            .rename(
                columns={
                    "index": "term",
                }
            )
        )


        # Replace internal factor names with
        # the corresponding AA labels.
        term_name_map = {
            "C(factor_1_present)":
                factor_1_aa,

            "C(factor_2_present)":
                factor_2_aa,

            (
                "C(factor_1_present):"
                "C(factor_2_present)"
            ):
                (
                    f"{factor_1_aa}:"
                    f"{factor_2_aa}"
                ),

            "Residual":
                "Residual",
        }

        anova_table[
            "term"
        ] = anova_table[
            "term"
        ].replace(
            term_name_map
        )

        anova_table.insert(
            0,
            "pair",
            pair_name,
        )

        anova_table.insert(
            1,
            "factor_1_AA",
            factor_1_aa,
        )

        anova_table.insert(
            2,
            "factor_2_AA",
            factor_2_aa,
        )

        anova_table.insert(
            3,
            "figure_group",
            figure_group,
        )

        anova_table.insert(
            4,
            "feature",
            feature_name,
        )

        all_anova_results.append(
            anova_table
        )


        safe_feature_name = (
            feature_name
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        individual_table_path = (
            ANOVA_TABLE_DIR
            / (
                f"{figure_group}_"
                f"{pair_name}_"
                f"{safe_feature_name}_ANOVA.csv"
            )
        )

        anova_table.to_csv(
            individual_table_path,
            index=False,
        )


        if figure_group == "main":
            figure_directory = (
                MAIN_FIGURE_DIR
            )

        else:
            figure_directory = (
                SUPPLEMENTARY_FIGURE_DIR
            )


        # Generate and save the interaction plot.
        fig, ax = plt.subplots(
            figsize=(3.5, 3)
        )

        interaction_plot(
            x=pair_data[
                "factor_1_present"
            ],
            trace=pair_data[
                "factor_2_present"
            ],
            response=pair_data[
                feature_name
            ],
            colors=[
                "#80A2C5",
                "#EC7166",
            ],
            markers=[
                "o",
                "v",
            ],
            markersize=10,
            markerfacecolor="none",
            linewidth=2,
            ax=ax,
        )

        ax.set_xlabel("")
        ax.set_ylabel("")

        ax.set_xticks(
            [
                0,
                1,
            ]
        )

        ax.set_xticklabels(
            [
                "Absence",
                "Presence",
            ],
            fontsize=10,
        )

        ax.tick_params(
            axis="y",
            labelsize=10,
        )

        legend = ax.get_legend()

        if legend is not None:
            legend.set_title(
                factor_2_aa
            )

            legend_texts = (
                legend.get_texts()
            )

            if len(legend_texts) == 2:
                legend_texts[0].set_text(
                    "Absence"
                )

                legend_texts[1].set_text(
                    "Presence"
                )

        fig.tight_layout()

        figure_path = (
            figure_directory
            / (
                f"{figure_group}_"
                f"{pair_name}_"
                f"{safe_feature_name}_"
                f"interaction_plot.png"
            )
        )

        fig.savefig(
            figure_path,
            dpi=600,
            bbox_inches="tight",
        )

        # Do not display all 44 plots in the notebook.
        plt.close(fig)


# Combine and save all ANOVA tables.
combined_anova_results = pd.concat(
    all_anova_results,
    ignore_index=True,
)

combined_anova_results.to_csv(
    ANOVA_OUTPUT_DIR
    / "two_way_ANOVA_all_pairs_all_features.csv",
    index=False,
)


# Combine and save the factor-combination counts.
combined_factor_counts = pd.concat(
    all_factor_counts,
    ignore_index=True,
)

combined_factor_counts.to_csv(
    ANOVA_OUTPUT_DIR
    / "two_way_ANOVA_factor_combination_counts.csv",
    index=False,
)


print(
    "\nAll two-way ANOVA analyses completed."
)

print(
    f"ANOVA pairs: "
    f"{len(ANOVA_PAIRS)}"
)

print(
    f"Features per pair: "
    f"{len(FEATURE_NAMES)}"
)

print(
    f"Total analyses: "
    f"{len(ANOVA_PAIRS) * len(FEATURE_NAMES)}"
)

print(
    f"Results saved to: "
    f"{ANOVA_OUTPUT_DIR.resolve()}"
)

Running SL: Baseline Ratio
Running SL: Z Score
Running SL: Per-event Impact
Running SL: Per-time Impact
Running DP: Baseline Ratio
Running DP: Z Score
Running DP: Per-event Impact
Running DP: Per-time Impact
Running DT: Baseline Ratio
Running DT: Z Score
Running DT: Per-event Impact
Running DT: Per-time Impact
Running GH: Baseline Ratio
Running GH: Z Score
Running GH: Per-event Impact
Running GH: Per-time Impact
Running HM: Baseline Ratio
Running HM: Z Score
Running HM: Per-event Impact
Running HM: Per-time Impact
Running LD: Baseline Ratio
Running LD: Z Score
Running LD: Per-event Impact
Running LD: Per-time Impact
Running MD: Baseline Ratio
Running MD: Z Score
Running MD: Per-event Impact
Running MD: Per-time Impact
Running MV: Baseline Ratio
Running MV: Z Score
Running MV: Per-event Impact
Running MV: Per-time Impact
Running SD: Baseline Ratio
Running SD: Z Score
Running SD: Per-event Impact
Running SD: Per-time Impact
Running TV: Baseline Ratio
Running TV: Z Score
Running TV: Per-e

In [6]:
# Summarize statistically significant
# interaction terms across all 44 analyses.

SIGNIFICANCE_THRESHOLD = 0.05

interaction_results = (
    combined_anova_results.loc[
        combined_anova_results[
            "term"
        ].str.contains(
            ":",
            regex=False,
            na=False,
        )
    ]
    .copy()
    .reset_index(drop=True)
)


expected_number_of_interactions = (
    len(ANOVA_PAIRS)
    * len(FEATURE_NAMES)
)

if (
    len(interaction_results)
    != expected_number_of_interactions
):
    raise ValueError(
        "Unexpected number of interaction terms. "
        f"Expected {expected_number_of_interactions}, "
        f"found {len(interaction_results)}."
    )


interaction_results[
    "significant"
] = (
    interaction_results[
        "PR(>F)"
    ]
    < SIGNIFICANCE_THRESHOLD
)


significant_interactions = (
    interaction_results.loc[
        interaction_results[
            "significant"
        ]
    ]
    .sort_values(
        [
            "pair",
            "feature",
        ]
    )
    .reset_index(drop=True)
)


number_significant = (
    interaction_results[
        "significant"
    ]
    .sum()
)

number_total = len(
    interaction_results
)

percentage_significant = (
    number_significant
    / number_total
    * 100
)


# Summarize significance by feature.
significance_by_feature = (
    interaction_results
    .groupby(
        "feature",
        sort=False,
    )
    .agg(
        total_interactions=(
            "significant",
            "size",
        ),
        significant_interactions=(
            "significant",
            "sum",
        ),
    )
    .reset_index()
)

significance_by_feature[
    "percentage_significant"
] = (
    significance_by_feature[
        "significant_interactions"
    ]
    / significance_by_feature[
        "total_interactions"
    ]
    * 100
)


# Summarize significance by AA pair.
significance_by_pair = (
    interaction_results
    .groupby(
        "pair",
        sort=False,
    )
    .agg(
        total_features=(
            "significant",
            "size",
        ),
        significant_features=(
            "significant",
            "sum",
        ),
    )
    .reset_index()
)

significance_by_pair[
    "percentage_significant"
] = (
    significance_by_pair[
        "significant_features"
    ]
    / significance_by_pair[
        "total_features"
    ]
    * 100
)


interaction_results.to_csv(
    ANOVA_OUTPUT_DIR
    / "two_way_ANOVA_interaction_results.csv",
    index=False,
)

significant_interactions.to_csv(
    ANOVA_OUTPUT_DIR
    / "two_way_ANOVA_significant_interactions.csv",
    index=False,
)

significance_by_feature.to_csv(
    ANOVA_OUTPUT_DIR
    / "two_way_ANOVA_significance_by_feature.csv",
    index=False,
)

significance_by_pair.to_csv(
    ANOVA_OUTPUT_DIR
    / "two_way_ANOVA_significance_by_pair.csv",
    index=False,
)


print(
    f"Significant interaction terms: "
    f"{int(number_significant)} "
    f"of {number_total} "
    f"({percentage_significant:.1f}%)"
)

print(
    "\nSignificant interactions:"
)

display(
    significant_interactions[
        [
            "pair",
            "feature",
            "term",
            "F",
            "PR(>F)",
        ]
    ]
)

print(
    "\nSummary by feature:"
)

display(
    significance_by_feature
)

print(
    "\nSummary by AA pair:"
)

display(
    significance_by_pair
)

Significant interaction terms: 40 of 44 (90.9%)

Significant interactions:


,pair,feature,term,F,PR(>F)
0,DP,Baseline Ratio,D:P,2209.525279,0.000000e+00
1,DP,Per-event Impact,D:P,3082.637801,0.000000e+00
2,DP,Per-time Impact,D:P,3481.993060,0.000000e+00
3,DP,Z Score,D:P,4511.676931,0.000000e+00
4,DT,Baseline Ratio,D:T,54.935881,1.255567e-13
5,DT,Per-event Impact,D:T,63.113940,1.972115e-15
6,DT,Per-time Impact,D:T,66.838296,2.983098e-16
7,DT,Z Score,D:T,50.642951,1.115743e-12
8,GH,Baseline Ratio,G:H,55.552584,9.176157e-14
9,GH,Per-event Impact,G:H,832.844020,2.412781e-182



Summary by feature:


,feature,total_interactions,significant_interactions,percentage_significant
0,Baseline Ratio,11,10,90.909091
1,Z Score,11,10,90.909091
2,Per-event Impact,11,10,90.909091
3,Per-time Impact,11,10,90.909091



Summary by AA pair:


,pair,total_features,significant_features,percentage_significant
0,SL,4,4,100.0
1,DP,4,4,100.0
2,DT,4,4,100.0
3,GH,4,4,100.0
4,HM,4,3,75.0
5,LD,4,1,25.0
6,MD,4,4,100.0
7,MV,4,4,100.0
8,SD,4,4,100.0
9,TV,4,4,100.0
